> 在线讲义：[第 25 章 · 提交、搬移与同步：从调用返回到结果可用](https://wikieden.github.io/ai-infra-from-constraints/lessons/25-submit-sync.html) · 本书《从约束推导 AI Infra》

# 第 25 章 · 提交、搬移与同步：从调用返回到结果可用

配套讲义：仓库里 `ai-infra/notes/lessons/25-submit-sync.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 例 5-1 | 字节 ÷ 带宽 | `transfer_ms(64 * MiB, 64e9)` | 1.05 / 0.52 ms |
| 第三步 例 5-2 | 依赖相加 | `ready_time(3, 8, 20, 4)` | 35 → 20 μs |
| 第三步 编译 | 摊销 | `amortized_ms(100, 1, 100)` | 101 / 2 ms |
| 第四步 驻留 | 取最小 | `resident_blocks(256, 128, 96 * 1024)` | 8 / 2 / 2 / 32 → 25% |
| 第四步 在途 | $RL/n$ | `inflight_bytes`、`inflight_have` | 15.2 KB vs 32 KiB；0.65、25 个 warp |
| 第四步 分工 | | `mma_count(128, 128, 64)` | 512 条；1.29 / 0.28 μs |

例 5-1、例 5-2、首次编译摊销、H100 单 SM 的驻留与在途字节、MMA 指令数、生产者 / 消费者 warp 时序取自 AII 第 5 章 §5.1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
MiB = 2**20

**函数 `transfer_ms`**：小工具。

In [ ]:
def transfer_ms(nbytes: float, rate: float) -> float:
    return nbytes / rate * 1e3

**函数 `ready_time`**：结果可用时刻（μs）：提交后加速器按依赖依次做 H2D、kernel、D2H。

In [ ]:
def ready_time(submit: float, h2d: float, kernel: float, d2h: float) -> float:
    """结果可用时刻（μs）：提交后加速器按依赖依次做 H2D、kernel、D2H。"""
    return submit + h2d + kernel + d2h

**函数 `amortized_ms`**：首次准备 prep、每次执行 run，调用 calls 次的平均耗时。

In [ ]:
def amortized_ms(prep: float, run: float, calls: int) -> float:
    """首次准备 prep、每次执行 run，调用 calls 次的平均耗时。"""
    return (prep + run * calls) / calls

**函数 `resident_blocks`**：各类资源各自允许的驻留线程块数（H100 一个 SM 的默认值）。

In [ ]:
def resident_blocks(threads: int, regs_per_thread: int, smem_bytes: int,
                    sm_threads: int = 2048, sm_regs: int = 65536, sm_smem: int = 233_472,
                    sm_blocks: int = 32, reserve: int = 1024) -> dict[str, int]:
    """各类资源各自允许的驻留线程块数（H100 一个 SM 的默认值）。"""
    return {
        "threads": sm_threads // threads,
        "regs": sm_regs // (threads * regs_per_thread),
        "smem": sm_smem // (smem_bytes + reserve),
        "blocks": sm_blocks,
    }

**函数 `occupancy`**：小工具。

In [ ]:
def occupancy(blocks: int, threads: int, max_warps: int = 64) -> float:
    return blocks * threads // 32 / max_warps

**函数 `inflight_bytes`**：每个 SM 需要保持的在途字节 = 带宽 × 延迟 / SM 数。

In [ ]:
def inflight_bytes(bw: float, latency_s: float, sms: int = 132) -> float:
    """每个 SM 需要保持的在途字节 = 带宽 × 延迟 / SM 数。"""
    return bw * latency_s / sms

**函数 `inflight_have`**：驻留 warp 能提供的在途字节：每条加载 32 线程 × 16 B = 512 B。

In [ ]:
def inflight_have(warps: int, loads_per_warp: int, bytes_per_load: int = 512) -> int:
    """驻留 warp 能提供的在途字节：每条加载 32 线程 × 16 B = 512 B。"""
    return warps * loads_per_warp * bytes_per_load

**函数 `mma_count`**：小工具。

In [ ]:
def mma_count(m: int, n: int, k: int, shape: tuple[int, int, int] = (16, 8, 16)) -> int:
    sm, sn, sk = shape
    return (m // sm) * (n // sn) * (k // sk)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：例 5-1，64 MiB 经 PCIe Gen5 x16（每方向 64 GB/s）")
print(f"  一次 {transfer_ms(64 * MiB, 64e9):.2f} ms；分两份各 {transfer_ms(32 * MiB, 64e9):.2f} ms")

In [ ]:
print("\n第二步：例 5-2，提交 3 + H2D 8 + kernel + D2H 4（μs）")
a, b = ready_time(3, 8, 20, 4), ready_time(3, 8, 5, 4)
print(f"  kernel 20 μs → {a:.0f} μs；kernel 5 μs → {b:.0f} μs；kernel 快 4 倍，整体快 {a / b:.2f} 倍")
print(f"  首次编译 100 ms + 每次 1 ms：1 次 {amortized_ms(100, 1, 1):.0f} ms，100 次平均 {amortized_ms(100, 1, 100):.1f} ms")

In [ ]:
print("\n第三步：H100 一个 SM，256 线程、128 寄存器、96 KiB 工作集")
r = resident_blocks(256, 128, 96 * 1024)
n = min(r.values())
print(f"  各资源允许 {r} → 驻留 {n} 块，占用率 {occupancy(n, 256):.0%}")
r2 = resident_blocks(256, 64, 32 * 1024)
n2 = min(r2.values())
print(f"  寄存器 64、共享内存 32 KiB：{r2} → {n2} 块，{occupancy(n2, 256):.0%}")
need, have = inflight_bytes(3.35e12, 600e-9), inflight_have(16, 4)
print(f"  在途字节：需要 {need / 1e3:.1f} KB，16 个 warp 提供 {have // 1024} KiB，覆盖 {have / need:.2f} 倍")
need2, have2 = inflight_bytes(3.35e12, 1000e-9), inflight_have(16, 2)
print(f"  延迟 1000 ns、每 warp 2 条：需要 {need2 / 1e3:.1f} KB，提供 {have2 // 1024} KiB，覆盖 {have2 / need2:.2f}，至少 {math.ceil(need2 / 1024)} 个 warp")

In [ ]:
print("\n第四步：矩阵指令与生产者 / 消费者 warp")
print(f"  128×128×64 块要 {mma_count(128, 128, 64)} 条 16×8×16 指令，每条 {2 * 16 * 8 * 16} FLOPs")
bw, pk = 3.35e12 / 132, 989.4e12 / 132
print(f"  每 SM {bw / 1e9:.1f} GB/s、{pk / 1e12:.2f} TFLOP/s：一块复制 {32 * 1024 / bw * 1e6:.2f} μs，计算 {2 * 128 * 128 * 64 / pk * 1e6:.2f} μs")

## 3. 代码里才看得到的

1. 228 KB 是 CUDA 文档的写法，实际是 233,472 B（228 × 1024）；每块 97 KiB = 99,328 B，两块后剩 34 KiB。
2. 在途字节的"需要"用 $10^3$ 制（15.2 KB），"提供"是 $2^{10}$ 制（32 KiB）；比值 2.15 是统一成字节后算的。
3. 每 SM 的带宽和峰值是整卡平分到 132 个 SM 的份额，假设这个线程块独占它所在的 SM。

## 4. 自己改一改

- 把 `resident_blocks` 的默认值换成 RTX PRO 6000（每 SM 1536 线程、64K 寄存器、100 KB 共享内存、24 块上限），同一个 kernel 驻留几块？
- 给 `inflight_have` 加一个"异步批量复制"版本：一个线程一次请求整块 16 KiB，需要几个在途块？
- 改 `ready_time`，把 H2D 和上一批的 kernel 重叠起来，连续 10 批要多久？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 服务员递单就转身，炒没炒好要另问；
> 原料带等"到了"，加工带等"用完"，漏一块牌子就死等；
> 填单三分钟，到手三十五；
> 教室座位、抽屉、书架，最先用完的定班级数；
> 班级够多，等材料时才有人在讲课。